In [845]:
import pycparser
import z3

In [846]:
def get_variable_name(d: dict[str, list[z3.z3.ExprRef]], name: str):
    return f"{name}_{len(d[name])}"

In [847]:
def translate_loop(s: z3.Solver, d: dict[str, list[z3.z3.ExprRef]], loop: pycparser.c_ast.For):
    translate_assignment(s,d,loop.init)
    translate_compound(s,d,loop.stmt)
    variable = loop.init.lvalue.name

    for i in range(translate_constant(loop.cond.right)):
        loop_variable = z3.Int(get_variable_name(d, variable))
        d[variable].append(loop_variable)
        value: z3.z3.ArithRef = d[variable][-2]
        s.add(loop_variable == value + 1)

        translate_compound(s,d,loop.stmt)

In [848]:
def translate_decl(s: z3.Solver, d: dict[str, list[z3.z3.ExprRef]], decl: pycparser.c_ast.Decl):
    match type(decl.type):
        case pycparser.c_ast.PtrDecl:
            variable = z3.Int(get_variable_name(d, decl.name))
            d[decl.name] = [variable]
        case pycparser.c_ast.TypeDecl:
            if decl.type.type.names[0] == 'int':
                variable = z3.Int(get_variable_name(d, decl.name))
                d[decl.name] = [variable]
            elif decl.type.type.names[0] == '_Bool':
                variable = z3.Bool(get_variable_name(d, decl.name))
                d[decl.name] = [variable]
            else:
                raise ValueError(f"Unknown TypeDecl type: {decl.type.type.names[0]}")
        case pycparser.c_ast.ArrayDecl:
            length = z3.Int(get_variable_name(d, decl.type.type.declname))
            d[decl.type.type.declname] = [length]
            s.add(length == translate_constant(decl.type.dim))
            return
        case _:
            raise ValueError(f"Unknown decl type: {decl.type}")
    if decl.init is not None:
        s.add(variable == translate_exp(s, d, decl.init))
    else :
        s.add(variable == 0)

In [849]:
def translate_constant(c: pycparser.c_ast.Constant):
    match c.type:
        case "int":
            return int(c.value)
        case "bool":
            return bool(c.value)
        case _:
            raise ValueError(f"Invalid constant type: {c.type}")

In [850]:
def translate_if(s: z3.Solver, d: dict[str, list[z3.z3.ExprRef]], _if: pycparser.c_ast.If, block_items_next: list):
    s.push()
    d_max = {k: len(v) for k, v in d.items()}
    s.add(translate_binary_op(s, d, _if.cond))
    translate_compound(s, d, _if.iftrue)
    translate_block_items(s, d, block_items_next)
    for k, v in d.items():
        if d_max.get(k) is not None:
            d[k] = v[:d_max[k]]
    s.pop()
    s.push()
    d_max = {k: len(v) for k, v in d.items()}
    s.add(z3.Not(translate_binary_op(s, d, _if.cond)))
    if _if.iffalse is not None:
        translate_compound(s, d, _if.iffalse)
    translate_block_items(s, d, block_items_next)
    for k, v in d.items():
        if d_max.get(k) is not None:
            d[k] = v[:d_max[k]]
    s.pop()

In [851]:
def translate_assignment(s: z3.Solver, d: dict[str, list[z3.z3.ExprRef]], assignment: pycparser.c_ast.Assignment):
    variable = z3.Int(get_variable_name(d, assignment.lvalue.name))
    d[assignment.lvalue.name].append(variable)
    s.add(variable == translate_exp(s, d, assignment.rvalue))

In [852]:
def translate_return(s: z3.Solver, d: dict[str, list[z3.z3.ExprRef]], _return: pycparser.c_ast.Return):
    translate_exp(s, d, _return.expr)

In [853]:
def translate_block_items(s: z3.Solver, d: dict[str, list[z3.z3.ExprRef]], block_items: list):
    for i, block_item in enumerate(block_items):
        match type(block_item):
            case pycparser.c_ast.If:
                translate_if(s, d, block_item, block_items[i+1:])
                return
            case pycparser.c_ast.For:
                translate_loop(s, d, block_item)
            case pycparser.c_ast.Decl:
                translate_decl(s, d, block_item)
            case pycparser.c_ast.Assignment:
                translate_assignment(s, d, block_item)
            case pycparser.c_ast.Return:
                translate_return(s, d, block_item)
            case _:
                raise ValueError(f"Invalid block_item type: {type(block_item)}")
    

In [854]:
def translate_compound(s: z3.Solver, d: dict[str, list[z3.z3.ExprRef]], compound: pycparser.c_ast.Compound):
    if compound.block_items is None:
        return
    translate_block_items(s, d, compound.block_items)

In [855]:
def translate_symbol(s: z3.Solver, d: dict[str, list[z3.z3.ExprRef]], symbol: pycparser.c_ast.Node):
    match symbol.__class__:
        case pycparser.c_ast.Constant:
            return translate_constant(symbol)
        case pycparser.c_ast.ID:
            return d[symbol.name][-1]
        case _:
            raise ValueError(f"Invalid symbol type: {symbol.__class__}")

In [856]:
def translate_arrayref(s: z3.Solver, d: dict[str, list[z3.z3.ExprRef]], exp: pycparser.c_ast.ArrayRef):
    length = d[exp.name.name][-1]
    value = translate_exp(s, d, exp.subscript)

    s.push()
    s.add(value >= length)
    if s.check() == z3.sat:
        m = s.model()
        print("traversing model...")
        for dec in m.decls():
            print("%s = %s" % (dec.name(), m[dec]))
        raise ValueError("Index out of bound")
    s.pop()

    s.push()
    s.add(value < 0)
    if s.check() == z3.sat:
        m = s.model()
        print ("traversing model...")
        for dec in m.decls():
            print("%s = %s" % (dec.name(), m[dec]))
        raise ValueError("Index out of bound")
    s.pop()

    return 0

In [857]:
def translate_exp(s: z3.Solver, d: dict[str, list[z3.z3.ExprRef]], exp: pycparser.c_ast.Node):
    match exp.__class__:
        case pycparser.c_ast.BinaryOp:
            return translate_binary_op(s, d, exp)
        case pycparser.c_ast.UnaryOp:
            return translate_unary_op(s, d, exp)
        case pycparser.c_ast.ID:
            return d[exp.name][-1]
        case pycparser.c_ast.Constant:
            return translate_constant(exp)
        case pycparser.c_ast.ArrayRef:
            return translate_arrayref(s, d, exp)
        case _:
            raise TypeError(f"Invalid expression type")

In [858]:
def translate_unary_op(s: z3.Solver, d: dict[str, list[z3.z3.ExprRef]], unaryOp: pycparser.c_ast.UnaryOp):
    match unaryOp.op:
        case "!":
            return z3.Not(translate_exp(s, d, unaryOp.expr))
        case "-":
            return 0 - translate_exp(s, d, unaryOp.expr)
        case "&":
            return 1
        case "*":
            s.push()
            if type(unaryOp.expr) == pycparser.c_ast.ID:
                variable = d[unaryOp.expr.name][-1]
                s.add( variable == 0)
            else:
                s.add( unaryOp.expr.value == 0)
            if s.check() == z3.sat:
                m = s.model()
                print ("traversing model...")
                for dec in m.decls():
                    print("%s = %s" % (dec.name(), m[dec]))
                raise ValueError("Deref of null pointer")
            s.pop()
            return 0
        case _:
            raise ValueError(f"Invalid condition type: {unaryOp.type}")


In [859]:
def translate_binary_op(s: z3.Solver, d: dict[str, list[z3.z3.ExprRef]], binaryOp: pycparser.c_ast.BinaryOp):
    match binaryOp.op:
        case "==":
            return translate_exp(s, d, binaryOp.left) == translate_exp(s, d, binaryOp.right)
        case "!=":
            return translate_exp(s, d, binaryOp.left) != translate_exp(s, d, binaryOp.right)
        case "<=":
            return translate_exp(s, d, binaryOp.left) <= translate_exp(s, d, binaryOp.right)
        case "<":
            return translate_exp(s, d, binaryOp.left) < translate_exp(s, d, binaryOp.right)
        case ">=":
            return translate_exp(s, d, binaryOp.left) >= translate_exp(s, d, binaryOp.right)
        case ">":
            return translate_exp(s, d, binaryOp.left) > translate_exp(s, d, binaryOp.right)
        case "/":
            s.push()
            s.add(translate_exp(s, d, binaryOp.right) == 0)
            if s.check() == z3.sat:
                m = s.model()
                print ("traversing model...")
                for dec in m.decls():
                    print("%s = %s" % (dec.name(), m[dec]))
                raise ValueError("Division by zero")
            s.pop()
            return translate_exp(s, d, binaryOp.left) / translate_exp(s, d, binaryOp.right)
        case "*":
            return translate_exp(s, d, binaryOp.left) * translate_exp(s, d, binaryOp.right)
        case "+":
            return translate_exp(s, d, binaryOp.left) + translate_exp(s, d, binaryOp.right)
        case "-":
            return translate_exp(s, d, binaryOp.left) - translate_exp(s, d, binaryOp.right)
        case _:
            raise ValueError(f"Invalid condition type: {binaryOp.type}")
    

In [860]:
from collections import defaultdict
ast: pycparser.FileAST = pycparser.parse_file("tests/wrong/deref_null.c")
compound: pycparser.c_ast.Compound = ast.ext[0].body
dico = defaultdict(list)
s = z3.Solver()
compound.block_items = compound.block_items
try:
    translate_compound(s, dico, compound)
except BaseException as e:
    print(e)

traversing model...
p_0 = 0
a_0 = 0
Deref of null pointer
